# Notebook 3 — Supervised Baseline (Experiment 1)
## BFI-Based Few-Shot Binary Occupancy Detection

**Goal:** Train a CNN classifier (encoder + linear head) on each device using standard
supervised learning. This establishes the per-device baseline (Experiment 1) and teaches
the complete PyTorch training loop before FSL complexity is added in Notebook 4.

**What this notebook does:**
1. Loads processed `.npy` tensors from Notebook 1 for each device (M7, X7, X300).
2. Wraps data in a PyTorch Dataset and DataLoader.
3. Defines `CNNClassifier` = `CNNEncoder` (Notebook 2) + linear head.
4. Implements `train_one_epoch` and `evaluate` — the core PyTorch supervised loop.
5. Reports per-fold and mean accuracy, precision, recall, F1, and AUC-PR for each device.
6. Saves the best encoder weights per device to `checkpoints/` for reuse in Notebook 4.
7. Trains a pooled supervised checkpoint on all three devices combined for use as Notebook 4 warm-start.

---
## 0. Configuration

All three devices (M7, X7, X300) are processed in a single run. Results are
reported per-device and as a combined summary table.
All settings match the hyperparameter spec in `bfi-fsl-overview.pdf` §6.2.

In [1]:
# Check if PyTorch is installed and if it has CUDA support
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# --user installs to your global user directory instead of the env.
import torch
print(torch.__version__)       # e.g., 2.x.x+cpu  ← bad, or 2.x.x+cu121 ← good
print(torch.version.cuda)      # None if CPU build


2.5.1+cu121
12.1


In [2]:
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import average_precision_score

# ── Paths ─────────────────────────────────────────────────────────────────────
PROCESSED_DIR  = Path('data/processed')   # Notebook 1 output folder
CHECKPOINT_DIR = Path('checkpoints')      # where trained encoder is saved
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
pool_ckpt_path = CHECKPOINT_DIR / 'nb3_supervised_pooled_best_encoder.pt'
npy_files = sorted(PROCESSED_DIR.rglob('*.npy'))

# ── Experiment target devices ────────────────────────────────────────────────
ALL_DEVICES  = ['M7', 'X7', 'X300']
TORCH_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Select file: set FILE_INDEX to pick a different one ──────────────────────
FILE_INDEX = 0  # <-- change this to select a different file
NB1_PATH = npy_files[FILE_INDEX]
print(f"\nUsing: [{FILE_INDEX}] {NB1_PATH}")


# ── Load and derive constants from array shape (W, T, K, C) ──────────────────
windows = np.load(NB1_PATH)  # shape: (W, T, K, C)
_, WINDOW_SIZE, TARGET_SUBCARRIERS, N_CHANNELS = windows.shape
LABEL_MAP = {'empty': 0, 'stationary': 1, 'moving': 1}

# ── Architecture — mirror Notebook 2 ─────────────────────────────────────────
EMBED_DIM   = 64
NUM_CLASSES = 2   # 0 = empty, 1 = occupied

# ── Training hyperparameters (bfi-fsl-overview.pdf §6.2) ─────────────────────
LEARNING_RATE = 1e-3
BATCH_SIZE    = 32
NUM_EPOCHS    = 100
PATIENCE      = NUM_EPOCHS  # effectively disables early stopping — see note below
RANDOM_SEED   = 67

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('Configuration loaded.')
print(f'  PyTorch device  : {TORCH_DEVICE}')
print(f'  BFI devices     : {ALL_DEVICES}')
print(f'  Processed dir   : {PROCESSED_DIR.resolve()}')
print(f'  Input shape     : (N, {N_CHANNELS}, {WINDOW_SIZE}, {TARGET_SUBCARRIERS})  (N, C, T, K)')
print(f'  LR={LEARNING_RATE}  batch={BATCH_SIZE}  epochs={NUM_EPOCHS}  patience={PATIENCE}')
print()
print('NOTE: PATIENCE is set to NUM_EPOCHS to effectively disable early stopping.')
print('Early stopping on training loss encourages overfitting. Instead, the full'
      ' fixed-epoch schedule is run and the best-loss checkpoint is restored.')



Using: [0] data\processed\M7\pvmatrix_empty-2_M7.npy
Configuration loaded.
  PyTorch device  : cuda
  BFI devices     : ['M7', 'X7', 'X300']
  Processed dir   : C:\Users\ghosty\Desktop\FSL-BFI\data\processed
  Input shape     : (N, 5, 5, 234)  (N, C, T, K)
  LR=0.001  batch=32  epochs=100  patience=100

NOTE: PATIENCE is set to NUM_EPOCHS to effectively disable early stopping.
Early stopping on training loss encourages overfitting. Instead, the full fixed-epoch schedule is run and the best-loss checkpoint is restored.


---
## 1. Data Loading Helper

Extended version of the Notebook 1 loader helper — adds a `position` field per window.
Position is inferred from the filename suffix:

| Filename pattern | Position |
|-----------------|----------|
| `p_vmatrix_*_M7.npy` (no `-N` suffix) | 1 |
| `p_vmatrix_*-2_M7.npy` | 2 |
| `p_vmatrix_*-3_M7.npy` | 3 |

The position index is the only key used for cross-validation splitting.

In [3]:
def load_processed(processed_dir: Path, devices=None):
    """
    Load all p*.npy window tensors from processed_dir (recursive search).

    Parameters
    ----------
    processed_dir : Path  Root of data/processed/ (contains M7/, X7/, X300/).
    devices       : list  Filter to specific devices, e.g. ['M7']. None = all.

    Returns
    -------
    windows : np.ndarray  (N, T, K, C)  channels-last
    labels  : np.ndarray  (N,) int64
    meta    : list[dict]  length N, keys: file, device, scenario, session, within_trace_idx
    """
    all_windows, all_labels, all_meta = [], [], []

    for fpath in sorted(processed_dir.glob("**/p*.npy")):
        stem     = fpath.stem[1:]   # strip leading 'p'
        scenario = next((k for k in LABEL_MAP if k in stem.lower()), None)
        device   = next((d for d in ['M7', 'X7', 'X300'] if stem.endswith(d)), None)

        if scenario is None or device is None:
            print(f"  WARNING: cannot parse {fpath.name}, skipping.")
            continue
        if devices and device not in devices:
            continue

        # Extract session number from filename suffix (-2 → 2, -3 → 3, no suffix → 1)
        m = re.search(r'-(\d+)_', stem)
        session = int(m.group(1)) if m else 1

        arr    = np.load(fpath)                              # (W, T, K, C)
        labels = np.full(arr.shape[0], LABEL_MAP[scenario], dtype=np.int64)

        all_windows.append(arr)
        all_labels.append(labels)
        for i in range(arr.shape[0]):
            all_meta.append({
                "file":             fpath.name,
                "device":           device,
                "scenario":         scenario,
                "session":          session,
                "within_trace_idx": i,
            })

    if not all_windows:
        raise FileNotFoundError(
            f"No p*.npy files found under {processed_dir}. "
            "Run Notebook 1 first."
        )

    return (np.concatenate(all_windows, axis=0),
            np.concatenate(all_labels,  axis=0),
            all_meta)


print('load_processed helper defined.')

load_processed helper defined.


---
## 2. PyTorch Dataset and DataLoader

`BFIDataset` converts channels-last NumPy arrays `(N, T, K, C)` to channels-first
tensors `(C, T, K)` per sample using `.permute(2, 0, 1)`. This matches the CNN's
expected input format without duplicating the full dataset in memory.

In [4]:
class BFIDataset(Dataset):
    """
    Wraps numpy window arrays and labels for PyTorch DataLoader.

    Input  : windows (N, T, K, C)  labels (N,)  — from Notebook 1 (channels-last)
    Returns: x (C, T, K) float32               — channels-first for Conv2d
             y int64 scalar label
    """

    def __init__(self, windows: np.ndarray, labels: np.ndarray):
        self.x = torch.from_numpy(windows).float()   # (N, T, K, C)
        self.y = torch.from_numpy(labels).long()     # (N,)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        x = self.x[idx]           # (T, K, C)
        x = x.permute(2, 0, 1)   # (C, T, K)  ← channels-first for Conv2d
        return x, self.y[idx]


print('BFIDataset defined.')

BFIDataset defined.


---
## 3. Model — CNN Encoder + Linear Classifier

`CNNEncoder` is reproduced verbatim from Notebook 2 (no changes).
`CNNClassifier` wraps it with one linear layer that maps the 64-dim embedding
to class logits.

```
x (N, C, T, K)
    └─ CNNEncoder ──→ embedding (N, 64)
                           └─ nn.Linear(64, 2) ──→ logits (N, 2)
```

The encoder is the only part transferred to Notebook 4; the linear head is
discarded after the supervised baseline.

In [5]:
class CNNEncoder(nn.Module):
    """
    4-layer 2D CNN encoder (Si-Fi / bfi-fsl-overview.pdf §5.3).
    Input : (N, C, T, K)  →  Output: (N, embed_dim)
    Identical to Notebook 2 — do not modify between notebooks.
    """

    def __init__(self, in_channels: int = N_CHANNELS, embed_dim: int = EMBED_DIM):
        super().__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True),
            )

        self.layer1 = conv_block(in_channels, embed_dim)
        self.layer2 = conv_block(embed_dim,   embed_dim)
        self.layer3 = conv_block(embed_dim,   embed_dim)
        self.layer4 = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x.flatten(start_dim=1)   # (N, embed_dim)


class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al., 2017) for binary classification.

    Down-weights easy examples and focuses training on hard, misclassified
    samples. This directly addresses the class-imbalance induced precision
    problem by preventing the model from becoming over-confident on the
    majority (occupied) class.

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    When gamma=0 this reduces to standard CrossEntropyLoss.
    """
    def __init__(self, alpha: torch.Tensor = None, gamma: float = 2.0):
        """
        Parameters
        ----------
        alpha : class weights tensor (one per class), or None.
        gamma : focusing parameter. gamma=2 is recommended by the paper.
        """
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt      = torch.exp(-ce_loss)  # predicted probability of true class
        focal   = ((1 - pt) ** self.gamma * ce_loss).mean()
        return focal


class CNNClassifier(nn.Module):
    """
    Supervised binary classifier: CNNEncoder + single linear head.

    Input : (N, C, T, K)
    Output: (N, num_classes)  raw logits  →  pass to nn.CrossEntropyLoss

    The encoder sub-module is saved separately as the Notebook 4 checkpoint.
    """

    def __init__(self,
                 in_channels: int = N_CHANNELS,
                 embed_dim:   int = EMBED_DIM,
                 num_classes: int = NUM_CLASSES):
        super().__init__()
        self.encoder = CNNEncoder(in_channels=in_channels, embed_dim=embed_dim)
        self.head    = nn.Linear(embed_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(x))   # (N, num_classes)


# ── Quick architecture check ──────────────────────────────────────────────────
_m   = CNNClassifier().to(TORCH_DEVICE)
_d   = torch.randn(2, N_CHANNELS, WINDOW_SIZE, TARGET_SUBCARRIERS).to(TORCH_DEVICE)
_out = _m(_d)
n_params = sum(p.numel() for p in _m.parameters())
del _m, _d

print('CNNClassifier and FocalLoss defined.')
print(f'  Logit shape  : {tuple(_out.shape)}  (expected (2, {NUM_CLASSES}))')
print(f'  Total params : {n_params:,}')
assert _out.shape == (2, NUM_CLASSES)
print('  PASS  logit shape correct.')


CNNClassifier and FocalLoss defined.
  Logit shape  : (2, 2)  (expected (2, 2))
  Total params : 114,114
  PASS  logit shape correct.


---
## 4. Training Utilities

Two functions implementing the core supervised PyTorch loop:

**`train_one_epoch`** — the 5-step pattern used in every PyTorch project:
```
for batch in loader:
    1. optimizer.zero_grad()       ← clear accumulated gradients
    2. logits = model(x)           ← forward pass
    3. loss   = criterion(logits, y)   ← compute scalar loss
    4. loss.backward()             ← backprop (fill .grad for every parameter)
    5. optimizer.step()            ← gradient descent step
```

**`evaluate`** — inference pass under `torch.no_grad()`, returns accuracy, precision,
recall, F1, and **AUC-PR** (area under the precision-recall curve). AUC-PR is the
preferred metric for imbalanced binary classification because it is threshold-independent
and reflects performance across all operating points:
\[
\text{Precision} = \frac{TP}{TP+FP}, \quad
\text{Recall} = \frac{TP}{TP+FN}, \quad
\text{F1} = \frac{2 \cdot P \cdot R}{P+R}
\]
A small epsilon (1e-8) prevents division by zero when a class is absent in tiny test sets.

In [6]:
def train_one_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device:    torch.device,
) -> float:
    """
    One full pass over the training set.  Returns mean training loss.
    """
    model.train()
    total_loss = 0.0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()                   # 1. clear gradients
        logits = model(x_batch)                 # 2. forward pass  → (N, 2)
        loss   = criterion(logits, y_batch)     # 3. compute loss
        loss.backward()                         # 4. backprop
        optimizer.step()                        # 5. update weights

        total_loss += loss.item() * len(y_batch)

    return total_loss / len(loader.dataset)


def evaluate(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: nn.Module,
    device:    torch.device,
) -> dict:
    """
    Evaluate model on a DataLoader.

    Returns dict with keys:
      loss, accuracy, precision, recall, f1, auc_pr  (floats)
      tp, fp, fn, tn                                  (ints, for confusion matrix)

    Precision / Recall / F1 target the positive class (label=1, occupied).
    AUC-PR is computed from the positive-class probabilities and is
    threshold-independent — the standard metric for imbalanced classification.
    """
    model.eval()
    all_preds, all_true, all_probs = [], [], []
    total_loss = 0.0

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits      = model(x_batch)
            total_loss += criterion(logits, y_batch).item() * len(y_batch)

            probs = F.softmax(logits, dim=1)[:, 1]      # P(occupied)
            preds = logits.argmax(dim=1)                 # hard prediction
            all_probs.extend(probs.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
            all_true.extend(y_batch.cpu().tolist())

    p = np.array(all_preds)
    t = np.array(all_true)
    eps = 1e-8

    tp = int(((p == 1) & (t == 1)).sum())
    fp = int(((p == 1) & (t == 0)).sum())
    fn = int(((p == 0) & (t == 1)).sum())
    tn = int(((p == 0) & (t == 0)).sum())

    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    f1        = 2 * precision * recall / (precision + recall + eps)
    accuracy  = float((p == t).mean())
    auc_pr    = average_precision_score(t, all_probs) if len(set(t)) > 1 else float('nan')

    return {
        'loss':      total_loss / len(loader.dataset),
        'accuracy':  accuracy,
        'precision': float(precision),
        'recall':    float(recall),
        'f1':        float(f1),
        'auc_pr':    float(auc_pr),
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
    }


print('train_one_epoch and evaluate defined.')


train_one_epoch and evaluate defined.


---
## 5. Per-Device Training Loop

For each device (M7, X7, X300):
1. Load data for that device.
2. Run 3-fold cross-validation (leave one session out).
3. Save the best encoder checkpoint.

For each fold:
1. Split windows by **session identity** (leave one session out as test).
2. **Undersample** large training sessions to match the smallest training session's
   window count — prevents a single recording (e.g., X7 session 3 with 18K frames)
   from dominating the training distribution.
3. Instantiate a fresh `CNNClassifier` (re-seeded each fold).
4. Train with Adam + **Focal Loss** (replaces CrossEntropyLoss). Focal Loss
   down-weights easy examples via gamma=2, forcing the model to focus on the
   harder (minority) class and directly improving precision.
5. Use **WeightedRandomSampler** instead of uniform shuffling — each mini-batch
   is balanced across classes, complementing the class-weight in the loss.
6. Run the full `NUM_EPOCHS` schedule (early stopping disabled — see note in §0).
7. Restore the best-loss weights and evaluate on the held-out test set.

**Session undersampling:** When one training session is significantly larger than
others, the model overfits to that session's specific position. Undersampling
ensures each training session contributes equally to the learned representation,
reducing position-specific bias. This is especially important for X7 where
session 3 contains 18,159 frames vs. ~350 for other sessions.

**WeightedRandomSampler** ensures each batch has ~50/50 class composition,
preventing batches from being 66% occupied (which biases gradient updates
toward the majority class).

**Focal Loss** further reduces the precision problem by down-weighting the
gradient contribution of already-correctly-classified examples.

In [7]:
# 3-fold split: hold out one session at a time (session identity split)
FOLDS = [
    {"name": "Fold 1", "test_session": 1},
    {"name": "Fold 2", "test_session": 2},
    {"name": "Fold 3", "test_session": 3},
]

# Collect results from all devices for final summary
all_device_results = {}  # {device_name: {'fold_results': [...], 'avg': {...}, 'best_overall': {...}}}

for DEVICE_NAME in ALL_DEVICES:
    print()
    print("=" * 72)
    print(f"  DEVICE: {DEVICE_NAME}")
    print("=" * 72)

    # ── Load data for this device ──────────────────────────────────────────────
    windows, labels, meta = load_processed(PROCESSED_DIR, devices=[DEVICE_NAME])

    sessions   = np.array([m["session"]   for m in meta])
    scenarios  = np.array([m["scenario"]  for m in meta])

    print(f"Loaded device: {DEVICE_NAME}")
    print(f"  windows shape : {windows.shape}  (N, T, K, C)")
    print(f"  class balance : {(labels==0).sum()} empty  {(labels==1).sum()} occupied")
    print()

    # ── Dynamic class-weighted loss weights ──────────────────────────────────
    counts    = np.bincount(labels)          # [n_empty, n_occupied]
    weights   = len(labels) / (2 * counts)  # inverse-frequency
    alpha     = torch.tensor(weights, dtype=torch.float).to(TORCH_DEVICE)
    print(f"Class weights — empty: {weights[0]:.3f}, occupied: {weights[1]:.3f}")

    fold_results = []
    best_overall = {"f1": -1.0, "fold": None, "state_dict": None}

    for fold in FOLDS:
        print()
        print("-" * 64)
        train_sessions = [s for s in [1, 2, 3] if s != fold["test_session"]]
        print(f"{fold['name']}  "
              f"test session [{fold['test_session']}]  "
              f"train sessions {train_sessions}")
        print("-" * 64)

        test_mask  = sessions == fold["test_session"]
        train_mask = ~test_mask

        # ── Undersample training sessions to balanced counts ──────────────────
        train_idxs = np.where(train_mask)[0]
        train_sess_ids = sessions[train_idxs]

        per_sess_counts = {s: int((train_sess_ids == s).sum()) for s in train_sessions}
        min_count = min(per_sess_counts.values())

        undersampled_idxs = []
        rng = np.random.default_rng(RANDOM_SEED)
        for s in train_sessions:
            s_idxs = train_idxs[train_sess_ids == s]
            if len(s_idxs) > min_count:
                s_idxs = rng.choice(s_idxs, size=min_count, replace=False)
            undersampled_idxs.extend(s_idxs.tolist())
        undersampled_idxs = np.array(undersampled_idxs)

        train_mask_balanced = np.zeros(len(windows), dtype=bool)
        train_mask_balanced[undersampled_idxs] = True

        print(f"  Session window counts (training): {per_sess_counts}")
        print(f"  Undersample min={min_count}, total train windows after: {undersampled_idxs.shape[0]}")

        train_ds = BFIDataset(windows[train_mask_balanced], labels[train_mask_balanced])
        test_ds  = BFIDataset(windows[test_mask],  labels[test_mask])

        print(f"  Train {len(train_ds):3d} windows  "
              f"class 0:{int((labels[train_mask_balanced]==0).sum())}  "
              f"class 1:{int((labels[train_mask_balanced]==1).sum())}")
        print(f"  Test  {len(test_ds):3d} windows  "
              f"class 0:{int((labels[test_mask]==0).sum())}  "
              f"class 1:{int((labels[test_mask]==1).sum())}")

        # ── WeightedRandomSampler for balanced mini-batches ──────────────────
        train_labels_np = labels[train_mask_balanced]
        class_counts    = np.bincount(train_labels_np)
        sample_weights  = 1.0 / class_counts[train_labels_np]
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True,
        )

        eff_train_bs = min(BATCH_SIZE, len(train_ds))
        eff_test_bs  = len(test_ds)

        train_loader = DataLoader(train_ds, batch_size=eff_train_bs,
                                  sampler=sampler, drop_last=False)
        test_loader  = DataLoader(test_ds,  batch_size=eff_test_bs,
                                  shuffle=False)

        # Fresh model, FocalLoss, and optimiser — re-seeded for reproducibility
        torch.manual_seed(RANDOM_SEED)
        model     = CNNClassifier(in_channels=N_CHANNELS,
                                  embed_dim=EMBED_DIM).to(TORCH_DEVICE)
        criterion = FocalLoss(alpha=alpha, gamma=2.0)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

        # Early stopping state (note: PATIENCE = NUM_EPOCHS, effectively disabled)
        best_loss    = float('inf')
        patience_ctr = 0
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        # Training loop
        print()
        print(f"  Training  max {NUM_EPOCHS} epochs  loss={type(criterion).__name__}  gamma=2.0")
        print(f"  {'Epoch':>6s}  {'Train Loss':>12s}  {'Patience':>10s}")
        print(f"  {'-'*34}")

        stopped_at = NUM_EPOCHS
        for epoch in range(1, NUM_EPOCHS + 1):
            train_loss = train_one_epoch(
                model, train_loader, optimizer, criterion, TORCH_DEVICE)

            if train_loss < best_loss:
                best_loss    = train_loss
                patience_ctr = 0
                best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                patience_ctr += 1

            if epoch == 1 or epoch % 10 == 0:
                print(f"  {epoch:>6d}  {train_loss:>12.4f}  "
                      f"{patience_ctr:>3d}/{PATIENCE}")

            if patience_ctr >= PATIENCE:
                stopped_at = epoch
                print(f"  {epoch:>6d}  {train_loss:>12.4f}  early stop")
                break

        print(f"  Best train loss {best_loss:.4f}  epoch {stopped_at}")

        # Restore best weights and evaluate
        model.load_state_dict(
            {k: v.to(TORCH_DEVICE) for k, v in best_state.items()})

        metrics = evaluate(model, test_loader, criterion, TORCH_DEVICE)
        fold_results.append({"fold": fold["name"], **metrics})

        print()
        print(f"  Confusion  TP={metrics['tp']}  FP={metrics['fp']}  "
              f"FN={metrics['fn']}  TN={metrics['tn']}")
        print(f"  Accuracy  {metrics['accuracy']:.3f}")
        print(f"  Precision {metrics['precision']:.3f}")
        print(f"  Recall    {metrics['recall']:.3f}")
        print(f"  F1        {metrics['f1']:.3f}")
        print(f"  AUC-PR    {metrics['auc_pr']:.3f}")

        if metrics["f1"] > best_overall["f1"]:
            best_overall = {
                "f1":         metrics["f1"],
                "fold":       fold["name"],
                "state_dict": best_state,
            }

    print()
    print("All 3 folds complete.")

    # ── Per-device results summary ────────────────────────────────────────────
    print()
    print(f'EXPERIMENT 1 RESULTS — {DEVICE_NAME}')
    print('=' * 62)
    print(f'  {"Fold":<10} {"Accuracy":>10} {"Precision":>10} '
          f'{"Recall":>10} {"F1":>10} {"AUC-PR":>10}')
    print(f'  {"-"*62}')

    for r in fold_results:
        print(f'  {r["fold"]:<10} {r["accuracy"]:>10.3f} '
              f'{r["precision"]:>10.3f} {r["recall"]:>10.3f} '
              f'{r["f1"]:>10.3f} {r["auc_pr"]:>10.3f}')

    print(f'  {"-"*62}')
    avg = {k: float(np.mean([r[k] for r in fold_results]))
           for k in ['accuracy', 'precision', 'recall', 'f1', 'auc_pr']}
    print(f'  {"Mean":<10} {avg["accuracy"]:>10.3f} '
          f'{avg["precision"]:>10.3f} {avg["recall"]:>10.3f} '
          f'{avg["f1"]:>10.3f} {avg["auc_pr"]:>10.3f}')
    print('=' * 62)

    if windows.shape[0] < 20:
        print()
        print(f'  *** Only {windows.shape[0]} windows — metrics are unreliable (test data).')
        print('  *** Rerun with production data for meaningful results.')

    # ── Save best encoder checkpoint ──────────────────────────────────────────
    print()
    ckpt_name = f'nb3_supervised_{DEVICE_NAME}_best_encoder.pt'
    ckpt_path = CHECKPOINT_DIR / ckpt_name

    # Isolate the encoder weights (strip 'encoder.' prefix)
    encoder_sd = {
        k[len('encoder.'):]: v
        for k, v in best_overall['state_dict'].items()
        if k.startswith('encoder.')
    }

    checkpoint = {
        'encoder_state_dict':    encoder_sd,
        'classifier_state_dict': best_overall['state_dict'],
        'best_fold':             best_overall['fold'],
        'best_f1':               best_overall['f1'],
        'device_name':           DEVICE_NAME,
        'config': {
            'n_channels':         N_CHANNELS,
            'embed_dim':          EMBED_DIM,
            'window_size':        WINDOW_SIZE,
            'target_subcarriers': TARGET_SUBCARRIERS,
            'num_classes':        NUM_CLASSES,
        },
        'avg_metrics':  avg,
        'fold_metrics': fold_results,
    }

    torch.save(checkpoint, ckpt_path)
    print(f'Checkpoint saved → {ckpt_path}')
    print(f'  Best fold : {best_overall["fold"]}  (F1 = {best_overall["f1"]:.3f})')
    print(f'  Mean AUC-PR across folds: {avg["auc_pr"]:.3f}')

    # Store results for final summary
    all_device_results[DEVICE_NAME] = {
        'fold_results': fold_results,
        'avg': avg,
        'best_overall': best_overall,
    }

print()
print("=" * 72)
print("  ALL DEVICES COMPLETE")
print("=" * 72)


  DEVICE: M7
Loaded device: M7
  windows shape : (648, 5, 234, 5)  (N, T, K, C)
  class balance : 218 empty  430 occupied

Class weights — empty: 1.486, occupied: 0.753

----------------------------------------------------------------
Fold 1  test session [1]  train sessions [2, 3]
----------------------------------------------------------------
  Session window counts (training): {2: 218, 3: 212}
  Undersample min=212, total train windows after: 424
  Train 424 windows  class 0:142  class 1:282
  Test  218 windows  class 0:74  class 1:144

  Training  max 100 epochs  loss=FocalLoss  gamma=2.0
   Epoch    Train Loss    Patience
  ----------------------------------
       1        0.0659    0/100
      10        0.0021    1/100
      20        0.0039   11/100
      30        0.0004    2/100
      40        0.0017    6/100
      50        0.0007   16/100
      60        0.0029   26/100
      70        0.0010   36/100
      80        0.0002    0/100
      90        0.0005    3/100
     1

---
## 6. Combined Results Summary

Mean metrics across the 3 folds for each device, and a grand mean across all devices.
These are the headline Experiment 1 numbers to report.

In [8]:
# ── Combined summary table across all devices ──────────────────────────────────
print()
print('EXPERIMENT 1 — COMBINED RESULTS (All Devices)')
print('=' * 82)
print(f'  {"Device":<8} {"Fold":<10} {"Accuracy":>10} {"Precision":>10} '
      f'{"Recall":>10} {"F1":>10} {"AUC-PR":>10}')
print(f'  {"-"*82}')

for dev in ALL_DEVICES:
    if dev in all_device_results:
        dev_data = all_device_results[dev]
        for r in dev_data['fold_results']:
            print(f'  {dev:<8} {r["fold"]:<10} {r["accuracy"]:>10.3f} '
                  f'{r["precision"]:>10.3f} {r["recall"]:>10.3f} '
                  f'{r["f1"]:>10.3f} {r["auc_pr"]:>10.3f}')
        # Print mean row
        avg = dev_data['avg']
        print(f'  {dev:<8} {"Mean":<10} {avg["accuracy"]:>10.3f} '
              f'{avg["precision"]:>10.3f} {avg["recall"]:>10.3f} '
              f'{avg["f1"]:>10.3f} {avg["auc_pr"]:>10.3f}')
        print(f'  {"-"*82}')

# Grand mean across all devices
grand_avg = {}
for metric in ['accuracy', 'precision', 'recall', 'f1', 'auc_pr']:
    values = [all_device_results[dev]['avg'][metric] for dev in ALL_DEVICES if dev in all_device_results]
    grand_avg[metric] = float(np.mean(values)) if values else float('nan')

print(f'  {"ALL":<8} {"Grand Mean":<10} {grand_avg["accuracy"]:>10.3f} '
      f'{grand_avg["precision"]:>10.3f} {grand_avg["recall"]:>10.3f} '
      f'{grand_avg["f1"]:>10.3f} {grand_avg["auc_pr"]:>10.3f}')
print('=' * 82)

print()
print('Reload encoder in Notebook 4:')
for dev in ALL_DEVICES:
    ckpt_name = f'nb3_supervised_{dev}_best_encoder.pt'
    print(f'  {dev}: ckpt = torch.load("checkpoints/{ckpt_name}")')
print()
print('--- Per-device training complete. Proceeding to pooled checkpoint. ---')



EXPERIMENT 1 — COMBINED RESULTS (All Devices)
  Device   Fold         Accuracy  Precision     Recall         F1     AUC-PR
  ----------------------------------------------------------------------------------
  M7       Fold 1          0.651      0.657      0.986      0.789      0.972
  M7       Fold 2          0.670      0.670      1.000      0.802      0.928
  M7       Fold 3          0.660      0.660      1.000      0.795      0.992
  M7       Mean            0.660      0.663      0.995      0.796      0.964
  ----------------------------------------------------------------------------------
  X7       Fold 1          0.800      0.824      0.964      0.889      0.910
  X7       Fold 2          0.585      0.642      0.855      0.734      0.826
  X7       Fold 3          0.973      0.983      0.989      0.986      0.999
  X7       Mean            0.786      0.816      0.936      0.869      0.912
  ----------------------------------------------------------------------------------
  X30

---
## 7. Pooled Supervised Checkpoint (Warm-Start for Notebook 4)

Trains a single supervised CNN on all windows from all three devices pooled together.
This checkpoint is used exclusively as the warm-start for Notebook 4 Experiments 2 and 3.
It is **not** an Experiment 1 result — no fold evaluation is done here.

Using a device-pooled warm-start ensures no single device has a pre-training advantage
when it becomes the held-out target in the leave-one-device-out evaluation.
The resulting file is saved as `checkpoints/nb3_supervised_pooled_best_encoder.pt`.

**Note:** This checkpoint's F1 metadata is evaluated on the training set and is
not a reported experimental result. Only the encoder weights matter for warm-start.

In [9]:
# Cell 7, first line
if pool_ckpt_path.exists():
    print(f"Pooled checkpoint already exists at {pool_ckpt_path}, skipping.")
else:
    # ── Load all three devices pooled ────────────────────────────────────────────
    windows_pool, labels_pool, _ = load_processed(PROCESSED_DIR, devices=['M7', 'X7', 'X300'])

    counts_pool = np.bincount(labels_pool)
    weights_pool = len(labels_pool) / (2 * counts_pool)
    alpha_pool = torch.tensor(weights_pool, dtype=torch.float).to(TORCH_DEVICE)

    pool_ds = BFIDataset(windows_pool, labels_pool)

    # Use WeightedRandomSampler for pooled training too
    pool_labels_np = labels_pool
    pool_class_counts = np.bincount(pool_labels_np)
    pool_sample_weights = 1.0 / pool_class_counts[pool_labels_np]
    pool_sampler = WeightedRandomSampler(
        weights=pool_sample_weights,
        num_samples=len(pool_sample_weights),
        replacement=True,
    )
    pool_loader = DataLoader(pool_ds, batch_size=BATCH_SIZE,
                             sampler=pool_sampler, drop_last=False)

    print(f"Pooled dataset: {len(pool_ds)} windows "
        f"(class 0={int((labels_pool==0).sum())} class 1={int((labels_pool==1).sum())})")
    print(f"Class weights — empty: {weights_pool[0]:.3f}, occupied: {weights_pool[1]:.3f}")

    # ── Train on full pooled dataset ──────────────────────────────────────────────
    torch.manual_seed(RANDOM_SEED)
    pool_model = CNNClassifier(in_channels=N_CHANNELS, embed_dim=EMBED_DIM).to(TORCH_DEVICE)
    pool_criterion = FocalLoss(alpha=alpha_pool, gamma=2.0)
    pool_optimizer = torch.optim.Adam(pool_model.parameters(), lr=LEARNING_RATE)

    best_pool_loss = float('inf')
    best_pool_state = {k: v.cpu().clone() for k, v in pool_model.state_dict().items()}
    pool_patience_ctr = 0
    pool_stopped_at = NUM_EPOCHS

    print(f"\nTraining pooled model (max {NUM_EPOCHS} epochs, FocalLoss gamma=2.0)")
    print(f" {'Epoch':>6s} {'Train Loss':>12s} {'Patience':>10s}")
    print(f" {'-'*34}")

    for epoch in range(1, NUM_EPOCHS + 1):
        loss = train_one_epoch(pool_model, pool_loader, pool_optimizer, pool_criterion, TORCH_DEVICE)
        if loss < best_pool_loss:
            best_pool_loss = loss
            pool_patience_ctr = 0
            best_pool_state = {k: v.cpu().clone() for k, v in pool_model.state_dict().items()}
        else:
            pool_patience_ctr += 1
        if epoch == 1 or epoch % 10 == 0:
            print(f" {epoch:>6d} {loss:>12.4f} {pool_patience_ctr:>3d}/{PATIENCE}")
        if pool_patience_ctr >= PATIENCE:
            pool_stopped_at = epoch
            print(f" {epoch:>6d} {loss:>12.4f}  early stop")
            break

    print(f" Best train loss: {best_pool_loss:.4f} (epoch {pool_stopped_at})")

    # ── Evaluate pooled model to get best_f1 for checkpoint ──────────────────────
    pool_model.load_state_dict({k: v.to(TORCH_DEVICE) for k, v in best_pool_state.items()})
    pool_eval_loader = DataLoader(pool_ds, batch_size=BATCH_SIZE, shuffle=False)
    pool_metrics = evaluate(pool_model, pool_eval_loader, pool_criterion, TORCH_DEVICE)

    # ── Save pooled encoder checkpoint ───────────────────────────────────────────
    pool_encoder_sd = {
        k[len('encoder.'):]: v
        for k, v in best_pool_state.items()
        if k.startswith('encoder.')
    }

    pool_ckpt_path = CHECKPOINT_DIR / 'nb3_supervised_pooled_best_encoder.pt'
    torch.save({
        'encoder_state_dict': pool_encoder_sd,
        'classifier_state_dict': best_pool_state,
        'device_name': 'pooled_M7_X7_X300',
        'best_loss': best_pool_loss,
        'best_fold': 'pooled',
        'best_f1': pool_metrics['f1'],  # training-set eval — not a reported result
        'config': {
            'n_channels': N_CHANNELS,
            'embed_dim': EMBED_DIM,
            'window_size': WINDOW_SIZE,
            'target_subcarriers': TARGET_SUBCARRIERS,
            'num_classes': NUM_CLASSES,
        },
    }, pool_ckpt_path)

    print(f"\nPooled checkpoint saved → {pool_ckpt_path}")
    print(f"  best_f1 (train-set eval, not a reported result): {pool_metrics['f1']:.3f}")
    print("Reload in Notebook 4:")
    print(f"  ckpt = torch.load('{pool_ckpt_path}')")
    print( "  encoder = CNNEncoder(in_channels=N_CHANNELS, embed_dim=EMBED_DIM)")
    print( "  encoder.load_state_dict(ckpt['encoder_state_dict'])")
    print()
    print("--- Notebook 3 complete. Proceed to Notebook 4: ProtoNet episodic training. ---")

Pooled dataset: 5729 windows (class 0=690 class 1=5039)
Class weights — empty: 4.151, occupied: 0.568

Training pooled model (max 100 epochs, FocalLoss gamma=2.0)
  Epoch   Train Loss   Patience
 ----------------------------------
      1       0.0740   0/100
     10       0.0036   0/100
     20       0.0017   0/100
     30       0.0023  10/100
     40       0.0012   2/100
     50       0.0007   3/100
     60       0.0011   3/100
     70       0.0005  13/100
     80       0.0003  23/100
     90       0.0007   2/100
    100       0.0004   1/100
 Best train loss: 0.0001 (epoch 100)

Pooled checkpoint saved → checkpoints\nb3_supervised_pooled_best_encoder.pt
  best_f1 (train-set eval, not a reported result): 1.000
Reload in Notebook 4:
  ckpt = torch.load('checkpoints\nb3_supervised_pooled_best_encoder.pt')
  encoder = CNNEncoder(in_channels=N_CHANNELS, embed_dim=EMBED_DIM)
  encoder.load_state_dict(ckpt['encoder_state_dict'])

--- Notebook 3 complete. Proceed to Notebook 4: ProtoNet epis